# HyLeakAI — U-Net surrogate training on Kaggle

Trains the U-Net surrogate mapping geology and operating stage to H2 saturation
and reservoir pressure fields, reproducing **Mao, Carbonero & Mehana (2025)**,
*JGR: Machine Learning and Computation*, `10.1029/2024JH000401`.

---

## Set these two things before running

1. **Settings → Accelerator → GPU T4 ×2.** Cell 1 refuses to run otherwise.
2. **Settings → Internet → On.** Required to fetch the dataset from Zenodo.

> **Do not pick P100.** Kaggle still offers it, but the P100 is compute
> capability 6.0 (Pascal) and recent PyTorch builds ship no Pascal kernels.
> `torch.cuda.is_available()` returns `True`, the run starts normally, and then
> dies on the first convolution with *"CUDA error: no kernel image is available
> for execution on the device"*. Cell 1 now checks the compute capability
> against the kernels actually compiled into this PyTorch build, and launches a
> real convolution to prove it works, so this fails in seconds instead of after
> the download. **T4 ×2 (sm_75) and L4 (sm_89) both work.**

## What this does

Fully self-contained — no repository, no manual uploads. It writes its own
source files, pulls the 12.38 GB dataset from Zenodo directly (Kaggle's
connection is fast, and the download is parallelised across 16 range requests),
converts it to ~3.9 GB of memmaps, verifies the conversion, deletes the raw
file, and trains.

## Sessions and resuming — read this before starting

A Kaggle GPU session is capped around 9 hours and the full 120-epoch run may not
fit in one. That is handled:

- Checkpoints are written **every epoch** to `/kaggle/working/checkpoints`,
  **atomically** (to `.tmp`, then moved), so being interrupted mid-save cannot
  destroy the checkpoint that exists to protect against interruption.
- **Three tiers are kept:** `_best.pt` (lowest validation loss), `_last.pt`
  (exact resume: model + optimizer + scheduler + history), and numbered
  `_epochNNN.pt` archival snapshots every 10 epochs (last 4 retained).
- Resume falls back through the snapshots newest-first, so a corrupt `_last.pt`
  costs you a few epochs rather than the whole run.

**To continue in a new session:** *Save Version → Save & Run All*, then in the
new session *Data → Add Input →* this notebook's output, set `PREV_OUTPUT` in
cell 2 to that path, and run all. Cell 9 detects the restored checkpoint and
resumes at the next epoch.

## Why U-Net-Small and not Large

The paper reports U-Net-Small **with the cyclic and distance channels** at 8.6%
pressure test error — level with U-Net-Large's 8.61% at 124M parameters and
35 GB. Without those channels Small collapses to 32.7%. The input
representation, not the parameter count, carries the accuracy. So this trains a
7.7M-parameter model on a free GPU and gives up almost nothing.

In [ ]:
# 1. Hardware check.
#
#    Two separate things can go wrong, and only the first is obvious:
#      (a) no GPU selected at all;
#      (b) a GPU is present but this PyTorch build has no kernels compiled for
#          its architecture. `torch.cuda.is_available()` returns True in that
#          case and the run dies on the first conv, ~30 minutes in.
#
#    (b) is live on Kaggle right now: the P100 is compute capability 6.0
#    (Pascal), and recent PyTorch builds dropped Pascal. Selecting P100 gives
#    "CUDA error: no kernel image is available for execution on the device".
#    Both are checked here, before anything expensive happens.
import subprocess

import torch

assert torch.cuda.is_available(), (
    "No GPU selected. Settings -> Accelerator -> GPU T4 x2, then re-run. "
    "Measured on CPU this model needs ~4 hours per epoch versus ~3-5 minutes on a T4."
)

name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
vram = torch.cuda.get_device_properties(0).total_memory / 2**30
arch_list = torch.cuda.get_arch_list()
print(f"GPU: {name}  compute capability sm_{major}{minor}  ({vram:.1f} GiB)")
print(f"torch {torch.__version__}, CUDA {torch.version.cuda}")
print(f"this build has kernels for: {arch_list}")

if f"sm_{major}{minor}" not in arch_list:
    raise SystemExit(
        f"\nThis PyTorch build has NO kernels for sm_{major}{minor} ({name}).\n"
        f"It has: {arch_list}\n\n"
        f"FIX: Settings -> Accelerator -> GPU T4 x2 (sm_75), then re-run.\n"
        f"     L4 (sm_89) also works. Avoid P100 (sm_60): recent PyTorch builds\n"
        f"     dropped Pascal support, and the failure only surfaces on the first\n"
        f"     convolution, long after training appears to have started.\n\n"
        f"Already-converted data in /kaggle/working survives the restart, so the\n"
        f"download and conversion cells will skip themselves."
    )

# Confirm end to end that a real kernel launches on this device.
try:
    _probe = torch.nn.Conv2d(2, 2, 3, padding=1).cuda()
    _ = _probe(torch.zeros(1, 2, 8, 8, device="cuda"))
    torch.cuda.synchronize()
    del _probe
    print("OK a convolution actually executes on this GPU.")
except Exception as exc:
    raise SystemExit(f"GPU present but unusable: {exc}\n"
                     f"FIX: Settings -> Accelerator -> GPU T4 x2, then re-run.")

# The paper used batch 128 at ~7.5 GB for U-Net-Small on a 40 GB A100.
# T4 is ~14.7 GiB, so 48 leaves comfortable headroom.
BATCH_SIZE = 96 if vram >= 20 else 48
print(f"Batch size: {BATCH_SIZE}")
print(subprocess.run(["df", "-h", "/kaggle/working"], capture_output=True, text=True).stdout)

In [ ]:
# 2. Paths. Set PREV_OUTPUT when resuming from a previous session.
from pathlib import Path

WORK = Path("/kaggle/working")
SCRATCH = Path("/kaggle/tmp"); SCRATCH.mkdir(exist_ok=True)   # not persisted
DATA = WORK / "data"                                          # converted arrays
CKPT = WORK / "checkpoints"                                   # checkpoints
RAW = SCRATCH / "data.mdb"                                    # 12.4 GB, deleted after convert

# Continuing a run: Data -> Add Input -> this notebook's previous output, then
# set the path below, e.g. "/kaggle/input/hileak-unet-training".
PREV_OUTPUT = None

for d in (DATA, CKPT):
    d.mkdir(parents=True, exist_ok=True)

if PREV_OUTPUT:
    import shutil
    prev = Path(PREV_OUTPUT)
    for sub, dst in (("checkpoints", CKPT), ("data", DATA)):
        s = prev / sub
        if s.exists():
            for f in s.iterdir():
                if not (dst / f.name).exists():
                    shutil.copy(f, dst / f.name)
                    print(f"restored {sub}/{f.name}")

print("checkpoints:", sorted(p.name for p in CKPT.glob("*.pt")) or "none")
print("data:", sorted(p.name for p in DATA.glob("*")) or "none")

In [ ]:
%%writefile hileak_core.py
"""HyLeakAI core: U-Net surrogate for underground hydrogen storage.

Self-contained single file for Kaggle — no repo checkout, no package install.
Mirrors src/config.py, src/models/unet.py and src/data/dataset.py.

Reproduces Mao, Carbonero & Mehana (2025), JGR: Machine Learning and
Computation, 10.1029/2024JH000401.
"""

from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset

# ---------------------------------------------------------------- constants

GRID = 128
DOMAIN_M = 7680.0                    # paper: 7,680 m length and width
CELL_M = DOMAIN_M / GRID             # 60 m
GRID_CENTER = (GRID - 1) / 2.0       # 63.5 — the central well
N_SIMS = 1000
N_TIMESTEPS = 60                     # t = 1..60; t=0 is pre-injection, identical
STEPS_PER_CYCLE = 6                  # 12 months / 2-month output interval
INJECTION_STEPS_PER_CYCLE = 3        # 6-month injection stage
P_INIT_BAR = 197.2                   # initial reservoir pressure
STATE_PRESSURE, STATE_SATURATION = 0, 1
SPLIT_SEED = 20260809
SPLIT_SIZES = {"train": 700, "val": 150, "test": 150}


def cycle_index(t: int) -> int:
    """+1 during injection, -1 during withdrawal (paper section 3.4).

    Verified against the simulations: on injection steps the well pressure
    exceeds 100.0% of the field, on withdrawal steps 0.0%.
    """
    return 1 if ((t - 1) % STEPS_PER_CYCLE) < INJECTION_STEPS_PER_CYCLE else -1


def cycle_number(t: int) -> int:
    return (t - 1) // STEPS_PER_CYCLE + 1


def simulation_splits(seed: int = SPLIT_SEED) -> dict[str, list[int]]:
    """Split BY SIMULATION (paper section 3.3: 700/150/150).

    A simulation's 60 timesteps are near-duplicates of one another. Splitting by
    sample would place timestep t in train and t+1 in test, leaking almost
    everything and inflating the reported error.
    """
    rng = np.random.default_rng(seed)
    ids = rng.permutation(N_SIMS)
    a, b = SPLIT_SIZES["train"], SPLIT_SIZES["val"]
    return {
        "train": sorted(ids[:a].tolist()),
        "val": sorted(ids[a : a + b].tolist()),
        "test": sorted(ids[a + b :].tolist()),
    }


# ---------------------------------------------------------------- model


class DoubleConv(nn.Module):
    """(conv 3x3 -> BN -> ReLU) x 2, as in the original U-Net."""

    def __init__(self, i: int, o: int, mid: int | None = None):
        super().__init__()
        mid = mid or o
        self.b = nn.Sequential(
            nn.Conv2d(i, mid, 3, padding=1, bias=False), nn.BatchNorm2d(mid), nn.ReLU(True),
            nn.Conv2d(mid, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(True),
        )

    def forward(self, x):
        return self.b(x)


class Down(nn.Module):
    def __init__(self, i: int, o: int):
        super().__init__()
        self.op = nn.Sequential(nn.MaxPool2d(2), DoubleConv(i, o))

    def forward(self, x):
        return self.op(x)


class Up(nn.Module):
    """Transposed-conv upsample, concatenate skip, DoubleConv."""

    def __init__(self, i: int, o: int):
        super().__init__()
        self.up = nn.ConvTranspose2d(i, i // 2, 2, stride=2)
        self.conv = DoubleConv(i, o)

    def forward(self, x, skip):
        x = self.up(x)
        dy, dx = skip.size(-2) - x.size(-2), skip.size(-1) - x.size(-1)
        if dy or dx:
            x = F.pad(x, [dx // 2, dx - dx // 2, dy // 2, dy - dy // 2])
        return self.conv(torch.cat([skip, x], dim=1))


class UNet(nn.Module):
    """Depth-4 U-Net. Embedding 32/64/128 gives the paper's Small/Medium/Large.

    On the 128x128 grid the bottleneck is 8x8.
    """

    def __init__(self, in_channels: int = 5, out_channels: int = 2, embedding: int = 32):
        super().__init__()
        e = embedding
        self.inc = DoubleConv(in_channels, e)
        self.d1, self.d2 = Down(e, e * 2), Down(e * 2, e * 4)
        self.d3, self.d4 = Down(e * 4, e * 8), Down(e * 8, e * 16)
        self.u1, self.u2 = Up(e * 16, e * 8), Up(e * 8, e * 4)
        self.u3, self.u4 = Up(e * 4, e * 2), Up(e * 2, e)
        self.outc = nn.Conv2d(e, out_channels, 1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.d1(x1)
        x3 = self.d2(x2)
        x4 = self.d3(x3)
        x5 = self.d4(x4)
        x = self.u1(x5, x4)
        x = self.u2(x, x3)
        x = self.u3(x, x2)
        x = self.u4(x, x1)
        return self.outc(x)

    @property
    def n_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


PAPER_SIZES = {"small": 32, "medium": 64, "large": 128}
PAPER_PARAMS_M = {"small": 7.7, "medium": 31.0, "large": 124.0}   # paper Table 1


def build_unet(size: str = "small", in_channels: int = 5, out_channels: int = 2) -> UNet:
    return UNet(in_channels, out_channels, PAPER_SIZES[size])


def assert_paper_parameter_counts(tol: float = 0.03) -> None:
    """Confirm the architecture matches the paper's reported weight counts.

    A mismatch means wrong depth, bilinear instead of transposed convolutions,
    or a different embedding progression — and any accuracy comparison to the
    paper would be meaningless. Runs before a single training step.
    """
    for s, want in PAPER_PARAMS_M.items():
        n = build_unet(s, 3, 1).n_parameters      # paper's 3-in/1-out configuration
        rel = abs(n / 1e6 - want) / want
        print(f"{'OK  ' if rel <= tol else 'FAIL'} U-Net-{s:6s} emb {PAPER_SIZES[s]:3d}  "
              f"{n / 1e6:7.2f}M  paper {want:6.1f}M  ({rel * 100:.1f}% off)")
        assert rel <= tol, f"U-Net-{s}: {n / 1e6:.2f}M vs paper {want}M"


# ---------------------------------------------------------------- loss


def relative_l2(pred: torch.Tensor, target: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    """Paper Eq. 1: ||T - P||_2 / ||T||_2, per sample, averaged over the batch.

    Training loss and reported test error are the same quantity, so the numbers
    compare directly with the paper's Table 4.
    """
    b = pred.shape[0]
    diff = (pred - target).reshape(b, -1).norm(dim=1)
    denom = target.reshape(b, -1).norm(dim=1).clamp_min(eps)
    return (diff / denom).mean()


class MultiHeadRelativeL2(nn.Module):
    """Weighted relative-L2 across pressure and saturation.

    The paper trains one model per state variable; we emit both from one
    two-channel head, which halves training cost. Both terms are scale-free so
    equal weights are a sound default.
    """

    def __init__(self, lambda_pressure: float = 1.0, lambda_saturation: float = 1.0):
        super().__init__()
        self.lp, self.ls = lambda_pressure, lambda_saturation

    def forward(self, pred, target):
        p = relative_l2(pred[:, STATE_PRESSURE], target[:, STATE_PRESSURE])
        s = relative_l2(pred[:, STATE_SATURATION], target[:, STATE_SATURATION])
        return self.lp * p + self.ls * s, {"pressure": p.detach(), "saturation": s.detach()}


# ---------------------------------------------------------------- data


def distance_to_well_map(grid: int = GRID) -> np.ndarray:
    """1 at the central well, 0 at the farthest corner (paper section 3.4).

    This channel is what makes U-Net-Small viable: the paper reports the small
    pressure model going from 32.7% error without cyclic+distance to 8.6% with
    them, level with U-Net-Large.
    """
    yy, xx = np.mgrid[0:grid, 0:grid].astype(np.float64)
    d = np.hypot(yy - GRID_CENTER, xx - GRID_CENTER)
    return (1.0 - d / d.max()).astype(np.float32)


class Normalizer:
    """Standardisation constants from stats.json.

    `log_permeability` defaults to deciding from the data: the measured field
    spans 1.03-738.8 mD (716x), so raw standardisation would leave a long right
    tail dominating the input. The paper standardises raw permeability; we make
    the choice explicit and logged rather than silent.
    """

    def __init__(self, stats: dict, log_permeability: bool | None = None):
        s = stats["stats"]
        self.poro_mean, self.poro_std = s["porosity"]["mean"], s["porosity"]["std"]
        perm = s["permeability"]
        if log_permeability is None:
            log_permeability = perm["max"] / max(perm["min"], 1e-30) > 100.0
        self.log_permeability = bool(log_permeability)
        src = s["log10_permeability"] if self.log_permeability else perm
        self.perm_mean, self.perm_std = src["mean"], src["std"]
        self.pres_mean, self.pres_std = s["pressure_bar"]["mean"], s["pressure_bar"]["std"]
        for n in ("poro_std", "perm_std", "pres_std"):
            if getattr(self, n) <= 0:
                raise ValueError(f"{n} is not positive; stats.json looks wrong")

    @classmethod
    def from_file(cls, path) -> "Normalizer":
        return cls(json.loads(Path(path).read_text()))

    def porosity(self, a):
        return (a - self.poro_mean) / self.poro_std

    def permeability(self, a):
        if self.log_permeability:
            a = np.log10(np.maximum(a, 1e-30))
        return (a - self.perm_mean) / self.perm_std

    def pressure(self, p_centred):
        """Input is CENTRED pressure as stored (P_bar - P_INIT_BAR)."""
        return (p_centred + P_INIT_BAR - self.pres_mean) / self.pres_std

    def pressure_inverse(self, p_std):
        return np.asarray(p_std) * self.pres_std + self.pres_mean

    def describe(self) -> str:
        return (f"Normalizer(log_permeability={self.log_permeability}, "
                f"pressure={self.pres_mean:.5g}+/-{self.pres_std:.4g} bar)")


class UHSDataset(Dataset):
    """One item = one (simulation, timestep) pair.

    Inputs  (5 ch): porosity, permeability, time, cyclic index, distance to well
    Targets (2 ch): pressure (standardised), saturation (raw, in [0,1])

    Saturation is deliberately not standardised — the paper found the transform
    gives no benefit because most of the field is exactly zero.

    Memmaps open lazily so each DataLoader worker gets its own handle.
    """

    def __init__(self, data_dir, sim_ids, normalizer: Normalizer,
                 use_cyclic: bool = True, use_distance: bool = True,
                 augment: bool = False):
        self.data_dir = Path(data_dir)
        self.sim_ids = list(sim_ids)
        self.normalizer = normalizer
        self.use_cyclic, self.use_distance = use_cyclic, use_distance
        self.augment = augment
        self._distance = distance_to_well_map()
        self._c = None
        self._s = None
        self.index = [(s, t) for s in self.sim_ids for t in range(1, N_TIMESTEPS + 1)]

    @property
    def constants(self):
        if self._c is None:
            self._c = np.load(self.data_dir / "constants.npy", mmap_mode="r")
        return self._c

    @property
    def states(self):
        if self._s is None:
            self._s = np.load(self.data_dir / "states.npy", mmap_mode="r")
        return self._s

    def __len__(self):
        return len(self.index)

    @property
    def in_channels(self) -> int:
        return 3 + int(self.use_cyclic) + int(self.use_distance)

    def build_input(self, sim: int, t: int) -> np.ndarray:
        c = np.asarray(self.constants[sim], np.float32)
        ch = [
            self.normalizer.porosity(c[0]),
            self.normalizer.permeability(c[1]),
            np.full((GRID, GRID), t / N_TIMESTEPS, np.float32),
        ]
        if self.use_cyclic:
            ch.append(np.full((GRID, GRID), float(cycle_index(t)), np.float32))
        if self.use_distance:
            ch.append(self._distance)
        return np.stack(ch).astype(np.float32)

    def build_target(self, sim: int, t: int) -> np.ndarray:
        st = np.asarray(self.states[sim, t - 1], np.float32)
        return np.stack([
            self.normalizer.pressure(st[STATE_PRESSURE]),
            st[STATE_SATURATION],
        ]).astype(np.float32)

    def build_input_tensor(self, sim: int, t: int) -> torch.Tensor:
        return torch.from_numpy(self.build_input(sim, t))

    def __getitem__(self, i: int):
        s, t = self.index[i]
        x = self.build_input(s, t)
        y = self.build_target(s, t)
        if self.augment:
            x, y = d4_transform(x, y)
        return torch.from_numpy(x), torch.from_numpy(y)


# ---------------------------------------------------------------- augmentation


def d4_transform(x: np.ndarray, y: np.ndarray, op: int | None = None):
    """Apply one of the 8 dihedral (D4) symmetries to input and target together.

    Why this is physically valid here, and not a generic image-augmentation
    reflex — every one of these has to hold, and all were measured:

      * The well sits at the exact grid centre (the 2x2 block at 63:65 on a
        128 axis), so it maps onto itself under all 8 operations.
      * The distance-to-well channel is D4-invariant to machine precision
        (verified: max |rot90(d) - d| = 0). The time and cyclic channels are
        uniform fields, so they are trivially invariant.
      * All four lateral boundaries are outflow, so no direction is special.
      * Permeability is a scalar per cell on a square grid, and the measured
        geology anisotropy is 0.6% (row-vs-column autocorrelation, 200 sims,
        lags 1-32) - i.e. the fields are isotropic, so a rotated realisation is
        still a plausible one rather than an off-distribution invention.
      * Rotational deviation of the ensemble-mean state field sits AT the
        finite-ensemble noise floor (pressure 0.0699 vs 0.0708), so the flow
        solution is rotation-equivariant within measurement error.

    Had the permeability been generated with a directional variogram, this
    would manufacture geology that never occurs and make things worse. It was
    not, so the 8x expansion is free.

    Uses torch's RNG rather than numpy's: PyTorch reseeds torch per DataLoader
    worker but does NOT reseed numpy, so numpy would hand every worker the
    same augmentation stream.
    """
    if op is None:
        op = int(torch.randint(0, 8, (1,)).item())
    k, flip = op % 4, op // 4
    if k:
        x = np.rot90(x, k, axes=(1, 2))
        y = np.rot90(y, k, axes=(1, 2))
    if flip:
        x = x[:, :, ::-1]
        y = y[:, :, ::-1]
    # rot90/slicing return views with negative strides; torch.from_numpy needs
    # a contiguous, positively-strided buffer.
    return np.ascontiguousarray(x), np.ascontiguousarray(y)


# ---------------------------------------------------------------- checkpoints


def save_checkpoint(path, model, opt, sched, epoch: int, history: list, meta: dict) -> None:
    """Write atomically: to a .tmp file, then move into place.

    An interruption mid-write — a session kill, a disconnect — must not be able
    to destroy the checkpoint that exists to protect against interruption.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save({"epoch": epoch, "model": model.state_dict(),
                "optimizer": opt.state_dict(), "scheduler": sched.state_dict(),
                "history": history, "meta": meta}, tmp)
    tmp.replace(path)


def load_checkpoint(path, model, opt=None, sched=None, device="cpu"):
    ck = torch.load(path, map_location=device, weights_only=False)
    model.load_state_dict(ck["model"])
    if opt is not None and "optimizer" in ck:
        opt.load_state_dict(ck["optimizer"])
    if sched is not None and "scheduler" in ck:
        sched.load_state_dict(ck["scheduler"])
    return ck


@torch.no_grad()
def evaluate(model, loader, device, amp: bool = False) -> dict[str, float]:
    """Mean per-sample relative L2 per state variable — the paper's metric."""
    model.eval()
    tot = {"pressure": 0.0, "saturation": 0.0}
    n = 0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, enabled=amp and device.type == "cuda"):
            p = model(x)
        tot["pressure"] += relative_l2(p[:, 0], y[:, 0]).item()
        tot["saturation"] += relative_l2(p[:, 1], y[:, 1]).item()
        n += 1
    return {k: v / max(n, 1) for k, v in tot.items()}

In [ ]:
%%writefile hileak_prep.py
"""Fetch the Mao et al. UHS dataset from Zenodo and convert it to memmaps.

Zenodo serves ~2-3 MB/s per connection but honours HTTP range requests, so
several concurrent connections cut wall time substantially (measured 9.6 MB/s
at 10 connections versus 2.6 MB/s single-stream).

The raw 12.38 GB LMDB is deleted once conversion is verified, keeping peak disk
well inside a Kaggle session's budget.

Layout produced:
    constants.npy  (1000, 2, 128, 128)      float32   ~131 MB
    states.npy     (1000, 60, 2, 128, 128)  float16   ~3.9 GB
    stats.json
"""

from __future__ import annotations

import hashlib
import json
import pickle
import threading
import time
from pathlib import Path

import numpy as np
import requests

URL = "https://zenodo.org/records/14029514/files/data.mdb?download=1"
TOTAL_BYTES = 12_380_934_144
MD5 = "6bc841f02ad3f40c9a8ef8ad187edf43"      # from the Zenodo record

GRID, N_SIMS, N_TIMESTEPS = 128, 1000, 60
P_INIT_BAR = 197.2


# ---------------------------------------------------------------- download


def download(dest, connections: int = 16, chunk_mb: int = 64) -> None:
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size == TOTAL_BYTES:
        print("Raw file already present, skipping download.")
        return

    cs = chunk_mb * 1024 * 1024
    chunks = [(s, min(s + cs, TOTAL_BYTES) - 1) for s in range(0, TOTAL_BYTES, cs)]
    with open(dest, "wb") as f:
        f.truncate(TOTAL_BYTES)          # preallocate so threads write disjoint offsets

    todo = list(range(len(chunks)))[::-1]
    lock = threading.Lock()
    done_bytes = [0]
    errors = []

    def worker():
        sess = requests.Session()
        while True:
            with lock:
                if not todo:
                    return
                i = todo.pop()
            a, b = chunks[i]
            written = 0
            try:
                r = sess.get(URL, headers={"Range": f"bytes={a}-{b}"},
                             stream=True, timeout=(15, 120))
                if r.status_code != 206:
                    raise RuntimeError(f"expected 206, got {r.status_code}")
                with open(dest, "r+b") as f:
                    f.seek(a)
                    for blk in r.iter_content(1 << 20):
                        if blk:
                            f.write(blk)
                            written += len(blk)
                if written != b - a + 1:
                    raise RuntimeError(f"short chunk {i}: {written}/{b - a + 1}")
                with lock:
                    done_bytes[0] += written
            except Exception as exc:
                with lock:
                    errors.append(f"chunk {i}: {exc}")
                    todo.append(i)          # requeue and retry
                time.sleep(2)

    threads = [threading.Thread(target=worker, daemon=True) for _ in range(connections)]
    t0 = time.time()
    for t in threads:
        t.start()
    while any(t.is_alive() for t in threads):
        time.sleep(10)
        d = done_bytes[0]
        rate = d / max(time.time() - t0, 1e-9)
        print(f"  {d / 2**30:6.2f}/{TOTAL_BYTES / 2**30:.2f} GiB  "
              f"{rate / 2**20:5.1f} MiB/s  "
              f"ETA {(TOTAL_BYTES - d) / max(rate, 1) / 60:5.1f} min", flush=True)
    for t in threads:
        t.join()

    if errors:
        print(f"{len(errors)} transient error(s) were retried.")

    print("Verifying md5...")
    h = hashlib.md5()
    with open(dest, "rb") as f:
        for blk in iter(lambda: f.read(8 << 20), b""):
            h.update(blk)
    got = h.hexdigest()
    assert got == MD5, f"md5 mismatch: {got} != {MD5}"
    print(f"OK md5 {got}")


# ---------------------------------------------------------------- convert


class _Stats:
    """float64 accumulators, so statistics are exact regardless of storage dtype."""

    def __init__(self):
        self.n = 0
        self.s = 0.0
        self.q = 0.0
        self.lo = np.inf
        self.hi = -np.inf

    def add(self, a):
        a = np.asarray(a, np.float64)
        self.n += a.size
        self.s += float(a.sum())
        self.q += float(np.square(a).sum())
        self.lo = min(self.lo, float(a.min()))
        self.hi = max(self.hi, float(a.max()))

    def out(self):
        m = self.s / self.n
        return {"count": self.n, "mean": m,
                "std": float(np.sqrt(max(self.q / self.n - m * m, 0.0))),
                "min": self.lo, "max": self.hi}


def _to_np(o):
    """Values are torch tensors in this dataset; normalise to numpy."""
    return o.detach().cpu().numpy() if hasattr(o, "detach") else np.asarray(o)


def convert(lmdb_path, out_dir) -> dict:
    """Stream the LMDB into compact memmaps.

    Two precision decisions, both deliberate:

    * Constants stay float32. Permeability is in millidarcy here (1.03-738.8 mD)
      so float16 would suffice, but a future revision in m^2 (~1e-13) would
      silently flush to zero. 131 MB is not worth that risk.

    * Pressure is stored as (P_bar - 197.2), centred on the initial reservoir
      pressure. Raw ~200 bar in float16 resolves to only 0.125 bar; centred, the
      values sit where the step is ~0.06 bar. Uses a known constant, so no extra
      pass over 12 GB is needed.

    Timestep 0 is dropped: the paper and README both state it is the
    pre-injection state, identical across all 1,000 simulations.

    Only the two channels the paper predicts are kept. The records actually
    carry an undocumented third channel which we measured to be ~99% collinear
    with pressure (aux ~= -1.17e-4 * (P - P_init)); it is not a training target.
    """
    import lmdb

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    if (out_dir / "states.npy").exists() and (out_dir / "stats.json").exists():
        print("Converted arrays already present, skipping conversion.")
        return json.loads((out_dir / "stats.json").read_text())

    env = lmdb.open(str(lmdb_path), subdir=False, readonly=True, lock=False,
                    readahead=False, max_readers=256)
    C = np.lib.format.open_memmap(out_dir / "constants.npy", mode="w+",
                                  dtype=np.float32, shape=(N_SIMS, 2, GRID, GRID))
    S = np.lib.format.open_memmap(out_dir / "states.npy", mode="w+", dtype=np.float16,
                                  shape=(N_SIMS, N_TIMESTEPS, 2, GRID, GRID))
    st = {k: _Stats() for k in ("porosity", "permeability", "log10_permeability",
                                "pressure_bar", "saturation")}

    t0 = time.monotonic()
    with env.begin() as txn:
        for sim in range(N_SIMS):
            poro, perm = [_to_np(a) for a in pickle.loads(txn.get(pickle.dumps(str(sim))))]
            C[sim, 0], C[sim, 1] = poro, perm
            st["porosity"].add(poro)
            st["permeability"].add(perm)
            st["log10_permeability"].add(np.log10(np.maximum(perm.astype(np.float64), 1e-30)))

            for t in range(1, N_TIMESTEPS + 1):
                v = pickle.loads(txn.get(pickle.dumps(f"{sim}-{t}")))
                p, s = _to_np(v[0]), _to_np(v[1])
                S[sim, t - 1, 0] = p.astype(np.float32) - P_INIT_BAR
                S[sim, t - 1, 1] = s
                st["pressure_bar"].add(p)
                st["saturation"].add(s)

            if (sim + 1) % 100 == 0:
                el = time.monotonic() - t0
                print(f"  {sim + 1}/{N_SIMS} sims  {el / 60:5.1f} min  "
                      f"ETA {(N_SIMS - sim - 1) / ((sim + 1) / el) / 60:5.1f} min", flush=True)

    env.close()
    C.flush()
    S.flush()
    summary = {
        "n_sims": N_SIMS, "n_timesteps": N_TIMESTEPS, "grid": GRID,
        "p_init_bar": P_INIT_BAR,
        "pressure_storage": "centred: stored = P_bar - P_INIT_BAR",
        "channels": {"constants": ["porosity", "permeability"],
                     "states": ["pressure_centred", "saturation"]},
        "stats": {k: v.out() for k, v in st.items()},
    }
    (out_dir / "stats.json").write_text(json.dumps(summary, indent=2))
    return summary


def verify(lmdb_path, out_dir, n_samples: int = 15, seed: int = 0) -> bool:
    """Round-trip random records against the LMDB.

    Tolerances are DERIVED from float16's resolution at each field's observed
    magnitude, not chosen to make the test pass. Pressure spans 82.7-293.8 bar,
    so centred values reach |114| bar where the float16 step is 0.125 — an error
    of ~0.03% of the excursion, roughly 300x below the surrogate's own ~8.6%.
    """
    import lmdb

    C = np.load(Path(out_dir) / "constants.npy", mmap_mode="r")
    S = np.load(Path(out_dir) / "states.npy", mmap_mode="r")
    assert C.shape == (N_SIMS, 2, GRID, GRID), C.shape
    assert S.shape == (N_SIMS, N_TIMESTEPS, 2, GRID, GRID), S.shape
    print(f"OK shapes: constants {C.shape}, states {S.shape}")

    env = lmdb.open(str(lmdb_path), subdir=False, readonly=True, lock=False)
    rng = np.random.default_rng(seed)
    worst_p = worst_s = 0.0
    with env.begin() as txn:
        for _ in range(n_samples):
            sim = int(rng.integers(0, N_SIMS))
            t = int(rng.integers(1, N_TIMESTEPS + 1))
            v = pickle.loads(txn.get(pickle.dumps(f"{sim}-{t}")))
            p, s = _to_np(v[0]), _to_np(v[1])
            got_p = np.asarray(S[sim, t - 1, 0], np.float32) + P_INIT_BAR
            got_s = np.asarray(S[sim, t - 1, 1], np.float32)
            worst_p = max(worst_p, float(np.abs(got_p - np.asarray(p, np.float32)).max()))
            worst_s = max(worst_s, float(np.abs(got_s - np.asarray(s, np.float32)).max()))

        # Confirm t=0 was excluded: stored[0] must match t=1, not t=0.
        p0 = _to_np(pickle.loads(txn.get(pickle.dumps("0-0")))[0])
        p1 = _to_np(pickle.loads(txn.get(pickle.dumps("0-1")))[0])
        first = np.asarray(S[0, 0, 0], np.float32) + P_INIT_BAR
        d0 = float(np.abs(first - np.asarray(p0, np.float32)).max())
        d1 = float(np.abs(first - np.asarray(p1, np.float32)).max())
    env.close()

    tol_p = float(np.spacing(np.float16(np.abs(S[:, :, 0]).max()))) / 2 * 1.5
    tol_s = float(np.spacing(np.float16(1.0))) / 2 * 1.5
    ok = worst_p <= tol_p and worst_s <= tol_s and d1 < d0
    print(f"{'OK  ' if worst_p <= tol_p else 'FAIL'} pressure   max|err| {worst_p:.3e} bar "
          f"(float16 limit {tol_p:.3e})")
    print(f"{'OK  ' if worst_s <= tol_s else 'FAIL'} saturation max|err| {worst_s:.3e} "
          f"(float16 limit {tol_s:.3e})")
    print(f"{'OK  ' if d1 < d0 else 'FAIL'} timestep 0 excluded "
          f"(d_to_t1 {d1:.3e} < d_to_t0 {d0:.3e})")
    assert ok, "conversion verification failed"
    return True

In [ ]:
%%writefile hileak_train.py
"""Training loop with three-tier checkpointing, built for interruptible sessions.

Optimiser settings follow the paper's Appendix Table A1: Adam, lr 1e-4,
weight decay 1e-5, learning rate halved every 50 epochs.

Checkpointing writes EVERY epoch and keeps three tiers, because a Kaggle GPU
session is capped well below a full 120-epoch run:

    _best.pt        lowest validation loss so far
    _last.pt        most recent epoch, for exact resume
    _epochNNN.pt    archival snapshots, the last few retained

`_best` and `_last` are rolling and overwrite each epoch; the numbered snapshots
are what let you return to a specific earlier epoch if a later one goes bad.

`target` selects the head configuration:

    "both"        one two-channel head for pressure and saturation (our default;
                  halves training cost, but the two targets conflict — pressure
                  is smooth, global and sign-flipping, saturation is local with
                  a sharp front, and a shared trunk must compromise)
    "pressure"    a single-headed model for pressure only
    "saturation"  a single-headed model for saturation only

The paper trains SEPARATE models per state variable. Running the two
single-headed variants tests whether the shared trunk is what puts our error
above the paper's.
"""

from __future__ import annotations

import json
import time
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader

from hileak_core import (
    STATE_PRESSURE,
    STATE_SATURATION,
    MultiHeadRelativeL2,
    Normalizer,
    UHSDataset,
    build_unet,
    evaluate,
    load_checkpoint,
    relative_l2,
    save_checkpoint,
    simulation_splits,
)

TARGET_CHANNEL = {"pressure": STATE_PRESSURE, "saturation": STATE_SATURATION}


@torch.no_grad()
def evaluate_single(model, loader, device, channel: int, amp: bool = False) -> float:
    """Relative L2 for a one-headed model predicting a single state variable."""
    model.eval()
    total, n = 0.0, 0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, enabled=amp and device.type == "cuda"):
            pred = model(x)
        total += relative_l2(pred[:, 0], y[:, channel]).item()
        n += 1
    return total / max(n, 1)


def train_unet(
    data_dir,
    ckpt_dir,
    size: str = "small",
    epochs: int = 120,
    batch_size: int = 64,
    lr: float = 1e-4,
    weight_decay: float | None = None,
    lr_halve_every: int = 50,
    augment: bool = True,
    workers: int = 2,
    amp: bool = True,
    overfit: int = 0,
    snapshot_every: int = 10,
    keep_snapshots: int = 4,
    resume: bool = True,
    seed: int = 20260809,
    target: str = "both",
):
    data_dir, ckpt_dir = Path(data_dir), Path(ckpt_dir)
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    if target not in ("both", "pressure", "saturation"):
        raise ValueError(f"target must be both/pressure/saturation, got {target!r}")

    # The paper's Appendix A1 uses a DIFFERENT weight decay per state variable:
    # 1e-4 for pressure, 1e-5 for saturation. The first run used 1e-5 for both,
    # leaving pressure — the worse head, val 0.193 against train 0.070 — ten
    # times under-regularised.
    #
    # A shared two-head trunk cannot honour both values, since one optimiser
    # applies one decay to the same weights. That, not the "conflicting targets"
    # story, is the concrete reason to train the heads separately: it is the
    # only configuration in which the paper's own hyperparameters are
    # expressible. For target="both" we keep 1e-5 so the comparison against the
    # first run stays like-for-like.
    if weight_decay is None:
        weight_decay = 1e-4 if target == "pressure" else 1e-5
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    torch.manual_seed(seed)
    np.random.seed(seed)

    norm = Normalizer.from_file(data_dir / "stats.json")
    print(f"Device: {device}")
    print(norm.describe())

    splits = simulation_splits()
    if overfit:
        # Deliberately tiny: train and evaluate on the SAME few simulations.
        # The only question this answers is whether the model can fit at all.
        ids = splits["train"][:overfit]
        # No augmentation here: the overfit probe asks "can the model fit at
        # all", and augmenting would make it fail for a reason unrelated to
        # architecture bugs, which is the only thing this mode tests.
        train_ds = UHSDataset(data_dir, ids, norm, augment=False)
        val_ds = UHSDataset(data_dir, ids, norm)
        weight_decay = 0.0
        augment = False
        print(f"OVERFIT MODE: {overfit} simulations, {len(train_ds)} samples, train == val")
    else:
        # Augment TRAIN ONLY. Augmenting validation would change what the
        # metric means and make it incomparable with the first run and the
        # paper.
        train_ds = UHSDataset(data_dir, splits["train"], norm, augment=augment)
        val_ds = UHSDataset(data_dir, splits["val"], norm)
        assert not set(train_ds.sim_ids) & set(val_ds.sim_ids), "train/val overlap"
        print(f"Simulations  train {len(train_ds.sim_ids)}  val {len(val_ds.sim_ids)}  "
              f"test {len(splits['test'])}")
        print(f"Samples      train {len(train_ds):,}  val {len(val_ds):,}")
        print(f"Augment      {'D4 (8x), train only' if augment else 'OFF'}   "
              f"weight decay {weight_decay:.0e} ({target})")

    kw = dict(num_workers=workers, pin_memory=(device.type == "cuda"),
              persistent_workers=workers > 0)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, **kw)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, **kw)

    single = target != "both"
    out_channels = 1 if single else 2
    channel = TARGET_CHANNEL[target] if single else None

    model = build_unet(size, train_ds.in_channels, out_channels).to(device)
    print(f"U-Net-{size}: {model.n_parameters / 1e6:.2f}M parameters, "
          f"{train_ds.in_channels} input channels, "
          f"{'single head -> ' + target if single else 'two heads -> pressure + saturation'}")

    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.StepLR(opt, step_size=lr_halve_every, gamma=0.5)
    loss_fn = MultiHeadRelativeL2()
    use_amp = amp and device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    # "_aug" is part of the tag, not cosmetic: resume=True keys off these paths,
    # so without it an augmented run would load the previous un-augmented
    # 120-epoch checkpoint and exit with "nothing to do" — silently reporting
    # the old result as the new one. Different hyperparameters, different files.
    tag = (f"unet_{size}" + (f"_{target}" if single else "")
           + ("_aug" if augment else "") + ("_overfit" if overfit else ""))
    best_path, last_path = ckpt_dir / f"{tag}_best.pt", ckpt_dir / f"{tag}_last.pt"
    meta = {"size": size, "in_channels": train_ds.in_channels, "use_cyclic": True,
            "use_distance": True, "batch_size": batch_size, "overfit": overfit, "target": target,
            "out_channels": out_channels, "augment": augment, "weight_decay": weight_decay,
            "lr": lr, "epochs_requested": epochs,
            "normalizer": norm.describe()}

    start, history, best_val = 0, [], float("inf")
    if resume:
        # Try _last, then archival snapshots newest-first. A resume should
        # degrade to "lose a few epochs", never to "start over".
        candidates = [last_path] + sorted(ckpt_dir.glob(f"{tag}_epoch*.pt"), reverse=True)
        for cand in candidates:
            if not cand.exists():
                continue
            try:
                ck = load_checkpoint(cand, model, opt, sched, device)
                start = ck["epoch"] + 1
                history = ck.get("history", [])
                best_val = min((h["val_total"] for h in history), default=float("inf"))
                print(f"RESUMED from {cand.name} at epoch {start} (best val {best_val:.4f})")
                break
            except Exception as exc:
                print(f"WARNING: could not load {cand.name}: {exc}")
        else:
            print("No usable checkpoint found; starting fresh.")

    if start >= epochs:
        print(f"Already trained through epoch {start - 1}; nothing to do.")
        return history

    print(f"\nTraining epochs {start}..{epochs - 1}\n", flush=True)
    for epoch in range(start, epochs):
        model.train()
        t0 = time.monotonic()
        run = {"pressure": 0.0, "saturation": 0.0}
        n = 0
        for x, y in train_loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                out = model(x)
                if single:
                    total = relative_l2(out[:, 0], y[:, channel])
                    parts = {"pressure": total.detach(), "saturation": total.detach()}
                else:
                    total, parts = loss_fn(out, y)
            scaler.scale(total).backward()
            scaler.step(opt)
            scaler.update()
            run["pressure"] += parts["pressure"].item()
            run["saturation"] += parts["saturation"].item()
            n += 1
        sched.step()

        if single:
            v = evaluate_single(model, val_loader, device, channel, amp=use_amp)
            val = {"pressure": v, "saturation": v}
            val_total = v
        else:
            val = evaluate(model, val_loader, device, amp=use_amp)
            val_total = val["pressure"] + val["saturation"]
        rec = {"epoch": epoch, "lr": opt.param_groups[0]["lr"],
               "train_pressure": run["pressure"] / max(n, 1),
               "train_saturation": run["saturation"] / max(n, 1),
               "val_pressure": val["pressure"], "val_saturation": val["saturation"],
               "val_total": val_total, "seconds": time.monotonic() - t0}
        history.append(rec)

        mark = ""
        if val_total < best_val:
            best_val = val_total
            save_checkpoint(best_path, model, opt, sched, epoch, history, meta)
            mark = "  <- best"
        save_checkpoint(last_path, model, opt, sched, epoch, history, meta)

        if snapshot_every > 0 and epoch % snapshot_every == 0:
            save_checkpoint(ckpt_dir / f"{tag}_epoch{epoch:03d}.pt",
                            model, opt, sched, epoch, history, meta)
            snaps = sorted(ckpt_dir.glob(f"{tag}_epoch*.pt"))
            for old in snaps[:-keep_snapshots]:
                old.unlink(missing_ok=True)

        (ckpt_dir / f"{tag}_history.json").write_text(json.dumps(history, indent=2))
        if single:
            print(f"epoch {epoch:3d}  train {target} {rec['train_pressure']:.4f}  |  "
                  f"val {target} {val['pressure']:.4f}  "
                  f"({rec['seconds']:.0f}s){mark}", flush=True)
        else:
            print(f"epoch {epoch:3d}  train P {rec['train_pressure']:.4f} "
                  f"S {rec['train_saturation']:.4f}  |  val P {val['pressure']:.4f} "
                  f"S {val['saturation']:.4f}  ({rec['seconds']:.0f}s){mark}", flush=True)

    print("\n" + "=" * 70)
    if overfit:
        f, l = history[0], history[-1]
        ok = l["train_pressure"] < 0.05 and l["train_saturation"] < 0.05
        print("OVERFIT TEST")
        print(f"  pressure   {f['train_pressure']:.4f} -> {l['train_pressure']:.4f}")
        print(f"  saturation {f['train_saturation']:.4f} -> {l['train_saturation']:.4f}")
        print(f"  {'PASS' if ok else 'NOT YET'}: a correct model drives both below 0.05 "
              f"on a handful of simulations.")
    else:
        b = min(history, key=lambda h: h["val_total"])
        if single:
            paper = 0.086 if target == "pressure" else 0.0577
            print(f"Best epoch {b['epoch']}: val {target} {b['val_pressure']:.4f}")
            print(f"Paper reference for {target}: {paper}")
        else:
            print(f"Best epoch {b['epoch']}: val pressure {b['val_pressure']:.4f}, "
                  f"saturation {b['val_saturation']:.4f}")
            print("Paper (U-Net-Large, 124M params, A100): pressure 0.0861, saturation 0.0577")
    print("=" * 70)
    return history

In [ ]:
# 6. Architecture check against the paper's Table 1.
#    A mismatch means wrong depth, bilinear instead of transposed convolutions,
#    or a different embedding progression -- and any accuracy comparison to the
#    paper would be meaningless. So this runs before a single training step.
import sys

sys.path.insert(0, "/kaggle/working")
from hileak_core import assert_paper_parameter_counts

assert_paper_parameter_counts()

In [ ]:
# 7. Fetch and convert the dataset. Skipped automatically if already present.
#    Download ~5-15 min on Kaggle's connection; conversion ~3-5 min.
!pip install -q lmdb 2>/dev/null

import json

import hileak_prep

if not (DATA / "states.npy").exists():
    hileak_prep.download(RAW, connections=16)
    print("\nConverting...")
    summary = hileak_prep.convert(RAW, DATA)
    print("\nVerifying...")
    hileak_prep.verify(RAW, DATA)
    RAW.unlink(missing_ok=True)          # reclaim 12.4 GB
    print("Raw LMDB deleted after verification.")
else:
    summary = json.loads((DATA / "stats.json").read_text())
    print("Using existing converted arrays.")

print()
for k, v in summary["stats"].items():
    print(f"  {k:20s} mean {v['mean']:12.5g}  std {v['std']:11.5g}  "
          f"range [{v['min']:.5g}, {v['max']:.5g}]")

In [ ]:
# 8. Overfit test: 4 simulations, expect both losses to fall toward 0.
#    Catches channel-ordering, normalisation and architecture bugs in minutes,
#    before committing hours to the real run. Do not skip this.
from hileak_train import train_unet

_ = train_unet(DATA, CKPT, size="small", epochs=40, batch_size=16,
               overfit=4, workers=2, snapshot_every=0, resume=False)

In [ ]:
# 9. RUN A. Full training with D4 augmentation. Re-run verbatim to resume.
#
# Why augmentation: run 1 ended at train pressure 0.070 / val 0.193 -- it fits
# training data BETTER than the paper's test error, so capacity was never the
# limit. The failure is generalisation to unseen geology. D4 (8 rotations and
# reflections) turns 42,000 training samples into an effective 336,000 at
# essentially no extra cost per epoch.
#
# Writes to unet_small_aug_* so it CANNOT resume from, or overwrite, run 1.
history = train_unet(
    DATA, CKPT,
    size="small",          # 7.7M params; matches Large WITH cyclic+distance
    epochs=120,
    batch_size=BATCH_SIZE,
    lr=1e-4,               # paper Appendix Table A1
    weight_decay=1e-5,     # matches run 1, so augmentation is the only variable
    lr_halve_every=50,     # "halved every 50 epochs"
    augment=True,
    workers=2,
    amp=True,
    snapshot_every=10,
    keep_snapshots=4,
    resume=True,
)

In [ ]:
# 10. Held-out TEST error. Run once, at the end.
import torch
from torch.utils.data import DataLoader

from hileak_core import (Normalizer, UHSDataset, build_unet, evaluate,
                         simulation_splits)

device = torch.device("cuda")
ck = torch.load(CKPT / "unet_small_aug_best.pt", map_location=device, weights_only=False)
norm = Normalizer.from_file(DATA / "stats.json")
splits = simulation_splits()

# The single most damaging possible bug in this project -- assert, do not assume.
for other in ("train", "val"):
    assert not set(splits["test"]) & set(splits[other]), f"test/{other} overlap"
print(f"OK splits disjoint. Test simulations: {len(splits['test'])}")

test = UHSDataset(DATA, splits["test"], norm)
model = build_unet("small", test.in_channels, 2).to(device)
model.load_state_dict(ck["model"])
m = evaluate(model, DataLoader(test, batch_size=BATCH_SIZE, num_workers=2), device)

print(f"\nTEST relative L2   pressure {m['pressure']:.4f}   saturation {m['saturation']:.4f}")
print("Paper (U-Net-Large, 124M params, A100): pressure 0.0861, saturation 0.0577")
print(f"Best checkpoint was epoch {ck['epoch']}")

In [ ]:
# 11. Training curves.
import json

import matplotlib.pyplot as plt

hist = json.loads((CKPT / "unet_small_aug_history.json").read_text())
ep = [h["epoch"] for h in hist]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, key, paper in ((ax[0], "pressure", 0.0861), (ax[1], "saturation", 0.0577)):
    a.plot(ep, [h[f"train_{key}"] for h in hist], label="train")
    a.plot(ep, [h[f"val_{key}"] for h in hist], label="val")
    a.axhline(paper, ls="--", c="k", lw=1, label="paper U-Net-Large")
    a.set_title(f"{key} relative L2"); a.set_xlabel("epoch")
    a.set_yscale("log"); a.legend(); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# 12. The two qualitative signatures from the paper.
#     If these are absent the model is wrong regardless of its headline number:
#       - pressure error SPIKES at each injection -> withdrawal transition (Fig. 8)
#       - saturation error concentrates at the PLUME FRONT (Fig. 10)
import numpy as np

from hileak_core import N_TIMESTEPS, cycle_index, relative_l2

ids = splits["test"][:40]
ds = UHSDataset(DATA, ids, norm)
err_p, err_s = np.zeros(N_TIMESTEPS), np.zeros(N_TIMESTEPS)
model.eval()
with torch.no_grad():
    for t in range(1, N_TIMESTEPS + 1):
        x = torch.stack([ds.build_input_tensor(s, t) for s in ids]).to(device)
        y = torch.stack([torch.from_numpy(ds.build_target(s, t)) for s in ids]).to(device)
        p = model(x)
        err_p[t - 1] = relative_l2(p[:, 0], y[:, 0]).item()
        err_s[t - 1] = relative_l2(p[:, 1], y[:, 1]).item()

steps = np.arange(1, N_TIMESTEPS + 1)
trans = [t for t in steps if t > 1 and cycle_index(t) == -1 and cycle_index(t - 1) == 1]
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(steps, err_p, marker="o", ms=3, label="pressure")
ax.plot(steps, err_s, marker="s", ms=3, label="saturation")
for i, t in enumerate(trans):
    ax.axvline(t, c="r", ls=":", lw=1,
               label="injection -> withdrawal" if i == 0 else None)
ax.set_xlabel("timestep (2 months each)"); ax.set_ylabel("relative L2")
ax.set_title("Temporal coherence (paper Fig. 8)"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

at = err_p[[t - 1 for t in trans]].mean()
elsewhere = err_p[[t - 1 for t in steps if t not in trans]].mean()
print(f"pressure error at transitions {at:.4f} vs elsewhere {elsewhere:.4f} "
      f"({at / elsewhere:.2f}x)")
print("PASS: reproduces the paper" if at > elsewhere else
      "CHECK: the paper reports a clear spike here; its absence is suspicious")

In [ ]:
# 13. Field comparison for one held-out simulation (paper Fig. 10).
sim = splits["test"][0]
show = [1, 3, 6, 12, 30, 60]
fig, axes = plt.subplots(4, len(show), figsize=(3 * len(show), 11))
with torch.no_grad():
    for j, t in enumerate(show):
        x = ds.build_input_tensor(sim, t)[None].to(device)
        y = ds.build_target(sim, t)
        p = model(x)[0].cpu().numpy()
        for r, (truth, pred, lab) in enumerate([(y[1], p[1], "saturation"),
                                                (y[0], p[0], "pressure")]):
            axes[2 * r, j].imshow(truth, cmap="viridis")
            axes[2 * r, j].set_title(f"{lab} truth  t={t}", fontsize=9)
            im = axes[2 * r + 1, j].imshow(truth - pred, cmap="RdBu_r")
            axes[2 * r + 1, j].set_title(f"{lab} error", fontsize=9)
            plt.colorbar(im, ax=axes[2 * r + 1, j], fraction=0.046)
for a in axes.ravel():
    a.set_xticks([]); a.set_yticks([])
plt.suptitle(f"Simulation {sim} (held out) - saturation error should ring the plume front")
plt.tight_layout(); plt.show()

## Optional experiment — two single-head models

Cells 9-13 train ONE model with two output channels, covering pressure and
saturation together. That halves training cost, but it is a deviation: the paper
trains a **separate model per state variable**.

Originally this section existed to test whether the two targets *conflict* —
pressure being smooth and global, saturation local with a sharp front. The run-1
history argues against that: train error reached 0.070 pressure / 0.037
saturation, better than the paper's *test* error, so a shared trunk clearly has
enough capacity to represent both fields. Head conflict is not the limiter.

The real reason to split is narrower and concrete. The paper's Appendix A1 uses
a **different weight decay per state variable** — 1e-4 for pressure, 1e-5 for
saturation. One optimiser over a shared trunk applies one decay to the same
weights, so a two-head model *cannot express the paper's own hyperparameters*.
Run 1 used 1e-5 for both, leaving pressure — the worse head — ten times
under-regularised on a model that is demonstrably overfitting.

Both cells below now train with D4 augmentation as well, so they are directly
comparable with Run A. Checkpoints are named separately
(`unet_small_pressure_aug_*`, `unet_small_saturation_aug_*`) and cannot
overwrite anything.

**Worth knowing before you spend the GPU hours:** for leakage screening the
two-head model already retains ~99% of the simulator's PR-AUC, so this affects
the reproduction claim and flux-magnitude accuracy far more than the risk
ranking. (That 99% figure is itself under review — see docs/FINDINGS.md.)

**How to read the result.** Compare against Run A, not against run 1:
- Run A alone closes most of the gap → augmentation was the fix; keep the
  cheaper two-head model.
- Only the split pressure model improves → the weight decay was the fix.
- Neither moves → the cause is batch 48 vs the paper's 128, or mixed precision,
  and the measured error should simply be reported as our honest number.

In [ ]:
# Single-head PRESSURE model. weight_decay defaults to the paper's 1e-4 here.
hist_p = train_unet(
    DATA, CKPT, size="small", target="pressure",
    epochs=120, batch_size=BATCH_SIZE, lr=1e-4,   # weight_decay=None -> 1e-4
    lr_halve_every=50, augment=True, workers=2, amp=True,
    snapshot_every=10, keep_snapshots=4, resume=True,
)

In [ ]:
# Single-head SATURATION model. weight_decay defaults to the paper's 1e-5.
hist_s = train_unet(
    DATA, CKPT, size="small", target="saturation",
    epochs=120, batch_size=BATCH_SIZE, lr=1e-4,   # weight_decay=None -> 1e-5
    lr_halve_every=50, augment=True, workers=2, amp=True,
    snapshot_every=10, keep_snapshots=4, resume=True,
)

In [ ]:
# Verdict: do two single-head models beat one two-head model?
import torch
from torch.utils.data import DataLoader

from hileak_core import (STATE_PRESSURE, STATE_SATURATION, Normalizer,
                         UHSDataset, build_unet, simulation_splits)
from hileak_train import evaluate_single

device = torch.device("cuda")
norm = Normalizer.from_file(DATA / "stats.json")
splits = simulation_splits()
test_ds = UHSDataset(DATA, splits["test"], norm)
loader = DataLoader(test_ds, batch_size=BATCH_SIZE, num_workers=2)

scores = {}
for target, channel in (("pressure", STATE_PRESSURE), ("saturation", STATE_SATURATION)):
    ck = torch.load(CKPT / f"unet_small_{target}_aug_best.pt", map_location=device,
                    weights_only=False)
    m = build_unet("small", test_ds.in_channels, 1).to(device)
    m.load_state_dict(ck["model"])
    scores[target] = evaluate_single(m, loader, device, channel)
    print(f"single-head {target:11s} test relative L2 {scores[target]:.4f}  "
          f"(best epoch {ck['epoch']})")

two_head = {"pressure": 0.1640, "saturation": 0.1101}   # measured in this project
paper = {"pressure": 0.086, "saturation": 0.0577}
print(f"\n{'variable':12s} {'two-head':>9} {'single-head':>12} {'change':>9} {'paper':>8}")
for k in ("pressure", "saturation"):
    print(f"{k:12s} {two_head[k]:>9.4f} {scores[k]:>12.4f} "
          f"{scores[k] - two_head[k]:>+9.4f} {paper[k]:>8.4f}")

if scores["pressure"] < two_head["pressure"] - 0.01:
    print("\nVERDICT: the shared trunk was the limiter — use separate models.")
else:
    print("\nVERDICT: splitting the heads did NOT help. The limiter is batch size "
          "or precision, and the cheaper two-head model should be kept.")

## Before you close the session

Running cells interactively persists **nothing**. `/kaggle/working` exists only
inside the live session. Close the tab or let it time out and the checkpoints
*and* the converted dataset are gone.

### Save Version → **Quick Save**

Quick Save commits the current session's `/kaggle/working` **without
re-executing anything** — which is exactly what you want once training has
already finished.

> **Do NOT choose "Save & Run All" here.** It restarts the notebook from cell 1
> — another 12.4 GB download and a full retrain — discarding the run you are
> trying to keep. "Save & Run All" is only for an unattended run you are
> deliberately launching.

Belt and braces: the editor's right-hand file browser also lets you download
`checkpoints/unet_small_aug_best.pt` (~93 MB) directly. Do both; it costs nothing.

To continue training later: new session → *Data → Add Input* → this notebook's
output → set `PREV_OUTPUT` in cell 2 → run all.

## What to expect

Roughly 3–5 min per epoch on a T4, so 120 epochs is about 7–10 hours, i.e.
two sessions. Target is relative L2 near **0.06–0.10 saturation** and
**0.09–0.13 pressure**.

The paper's best (U-Net-Large, 124M parameters, A100) is **0.0577** and
**0.0861**. Landing near 0.10 with a 7.7M-parameter model on a free GPU is a
legitimate reproduction and should be reported as exactly that — not rounded up
to the paper's numbers.